# Microsoft Agent Framework Harness Lane

This notebook is Demo Part 2: the same incident-response harness use case, implemented with Microsoft Agent Framework (MAF).

The point is not “MAF beats Strands” or “framework X wins.” The point for a Microsoft-shop audience is more useful:

> Harness engineering is becoming productized. Microsoft Agent Framework gives us enterprise-native building blocks for orchestration, sessions, memory, middleware, human approval, and packaged harness agents. We still have to define our domain quality contract: evidence, runbook compliance, safety policy, completion criteria, and escalation rules.

## 1. Fresh Colab Setup

Run this after cloning the repo in Colab. It installs the demo project plus Microsoft Agent Framework packages.

If you use Ollama Cloud models through the local Ollama endpoint, keep `ollama serve` running in the single Colab terminal and sign in there. If your Colab can reach an OpenAI-compatible cloud endpoint directly, you can point `MAF_BASE_URL` to that endpoint instead.

In [ ]:
# Replace this with your pushed GitHub repository URL before the management demo.
REPO_URL = "https://github.com/YOUR_ORG/ollama-harness-engineering-demo.git"

import os
from pathlib import Path

if not Path("/content/ollama-harness-engineering-demo").exists():
    !git clone "$REPO_URL" /content/ollama-harness-engineering-demo

%cd /content/ollama-harness-engineering-demo
!python -m pip install -q -r requirements.txt
!python -m pip install -q agent-framework agent-framework-openai agent-framework-orchestrations
!python -m pip install -q -e .

## 2. Load The Same Incident Scenario And Evaluator

This is what keeps the comparison fair. MAF will generate the response, but the scoring contract is the same deterministic contract used for no-harness, weak-harness, hand-built harness, Strands, and DeepSeek.

In [ ]:
import json
import os
from pprint import pprint
from IPython.display import Markdown, display

from harness_demo.domain import Lane
from harness_demo.live import score_freeform_answer
from harness_demo.rules import evaluate_rules
from harness_demo.scenarios import load_incident_scenario
from harness_demo.colab_display import (
    render_comparison_markdown,
    render_executive_findings_markdown,
    render_management_summary_markdown,
    render_memory_markdown,
    render_model_output_html,
    render_rule_findings_markdown,
)
from harness_demo.summarizer import critique_groundedness_with_ollama, summarize_result_with_ollama

scenario = load_incident_scenario("incident-response")

print("Incident:")
pprint(scenario.incident)
print("\nQuality contract used by deterministic scorer:")
pprint(scenario.expected)

## 3. Configure Ollama Cloud For MAF

MAF talks to model providers through clients. For Ollama Cloud in Colab, the most reliable route is usually:

1. Terminal: `ollama serve`
2. Terminal: `ollama signin`
3. Notebook: MAF uses `http://127.0.0.1:11434/v1/` as an OpenAI-compatible endpoint and calls a `:cloud` model.

Ollama’s public docs state that cloud models can be run through local Ollama after sign-in, and that Ollama exposes an OpenAI-compatible `/v1/chat/completions` API. If you later use Azure OpenAI or Foundry, only the client configuration changes; the harness contract stays the same.

In [ ]:
from getpass import getpass

# Option A: local Ollama endpoint, backed by Ollama Cloud after `ollama signin` in the terminal.
MAF_BASE_URL = os.environ.get("MAF_BASE_URL", "http://127.0.0.1:11434/v1/")

# For local Ollama, the API key can be any non-empty placeholder.
# If you point MAF_BASE_URL to a real cloud OpenAI-compatible endpoint, use its actual API key.
MAF_API_KEY = os.environ.get("MAF_API_KEY") or os.environ.get("OLLAMA_API_KEY") or "ollama"
os.environ["MAF_API_KEY"] = MAF_API_KEY

MAF_MODEL = os.environ.get("MAF_MODEL", "deepseek-v4-flash:cloud")
SUMMARY_MODEL = os.environ.get("SUMMARY_MODEL", "gpt-oss:20b")

print("MAF base URL:", MAF_BASE_URL)
print("MAF model:", MAF_MODEL)
print("Summary model:", SUMMARY_MODEL)

## 4. MAF Harness Mapping For This Use Case

| Harness concern | MAF construct in this notebook | Business meaning |
| --- | --- | --- |
| Task decomposition | Separate evidence, runbook, memory, planner, and reviewer agents | Different specialists produce inspectable intermediate artifacts |
| Context sharing | Sequential orchestration passes prior agent messages forward | Later agents can use earlier findings without one giant prompt |
| Tool/context filtering | Each agent receives only the context relevant to its role | Reduces tool/context bloat and limits accidental invention |
| Goal loop | Deterministic evaluator plus bounded repair attempts | Final output must satisfy the quality contract, not just sound good |
| Safety/HITL | Reviewer findings and HITL packet before unsafe/low-score result | Humans approve exceptions, not every model suggestion |
| Evaluation | Same scorer and rule engine as other lanes | Apples-to-apples comparison across frameworks |

## 5. Build A Microsoft Agent Framework Multi-Agent Workflow

This uses MAF `Agent` objects and `SequentialBuilder`. Each agent is intentionally narrow:

- evidence agent: logs and incident facts
- runbook agent: approved actions and forbidden actions
- memory agent: prior incident lessons
- planner agent: structured JSON response
- reviewer agent: checks the candidate against the quality contract

The deterministic scorer still has final authority. That distinction matters: the framework orchestrates work, while our quality contract decides whether the work is acceptable.

In [ ]:
import asyncio
from typing import Any, cast

from agent_framework import Agent, AgentResponse
from agent_framework.openai import OpenAIChatCompletionClient
from agent_framework.orchestrations import SequentialBuilder

client = OpenAIChatCompletionClient(
    base_url=MAF_BASE_URL,
    api_key=MAF_API_KEY,
    model=MAF_MODEL,
)


def as_text(response: Any) -> str:
    if response is None:
        return ""
    if isinstance(response, str):
        return response
    text = getattr(response, "text", None)
    if text:
        return str(text)
    messages = getattr(response, "messages", None)
    if messages:
        return "\n\n".join(str(getattr(message, "text", message)) for message in messages)
    return str(response)


evidence_agent = Agent(
    client=client,
    name="maf-evidence-agent",
    instructions=(
        "You are the evidence agent. Use only the incident ticket and logs. "
        "Return exact facts that support the likely cause and downstream symptoms. "
        "Do not propose fixes."
    ),
)

runbook_agent = Agent(
    client=client,
    name="maf-runbook-agent",
    instructions=(
        "You are the runbook agent. Use only the runbook section. "
        "Return approved actions, thresholds, rollback guidance, and forbidden actions. "
        "Do not invent operational systems, owners, dashboards, or thresholds."
    ),
)

memory_agent = Agent(
    client=client,
    name="maf-memory-agent",
    instructions=(
        "You are the memory agent. Use only prior incident memory. "
        "Return lessons relevant to this incident, especially actions that previously made the incident worse."
    ),
)

planner_agent = Agent(
    client=client,
    name="maf-planner-agent",
    instructions=(
        "You are the incident planner. Use the prior agent messages as shared context. "
        "Return JSON only with these fields: likely_cause, evidence, safe_next_action, "
        "rollback_plan, customer_impact, open_questions. Every action must be supported by "
        "the runbook or prior agent evidence. Do not recommend forbidden actions."
    ),
)

reviewer_agent = Agent(
    client=client,
    name="maf-reviewer-agent",
    instructions=(
        "You are the safety and quality reviewer. Review the planner JSON against the quality contract. "
        "Call out missing fields, missing runbook steps, unsupported claims, forbidden actions, and contradictions. "
        "Do not rewrite the final plan unless it is needed to remove a safety issue."
    ),
)

maf_workflow = SequentialBuilder(
    participants=[evidence_agent, runbook_agent, memory_agent, planner_agent, reviewer_agent],
    output_from="all",
).build()

maf_task = f"""
Incident ticket:
{json.dumps(scenario.incident, indent=2)}

Logs:
{scenario.logs}

Runbook:
{scenario.runbook}

Prior incident memory:
{scenario.prior_memory}

Quality contract:
{json.dumps(scenario.expected, indent=2)}

Run the multi-agent incident workflow. The planner must produce the final JSON plan. The reviewer must identify any gaps.
""".strip()

print("MAF workflow ready with agents:")
for agent in [evidence_agent, runbook_agent, memory_agent, planner_agent, reviewer_agent]:
    print("-", agent.name)

## 6. Run MAF Once And Score It

This is the first-pass MAF result. Do not worry if the score is imperfect. A useful management demo shows that the harness can detect gaps and explain what must be repaired.

In [ ]:
maf_first_pass_output = ""
maf_first_pass_result = None

try:
    workflow_result = await maf_workflow.run(maf_task)
    outputs = workflow_result.get_outputs()
    parts = []
    for output in outputs:
        response = cast(AgentResponse, output)
        parts.append(as_text(response))
    maf_first_pass_output = "\n\n".join(part for part in parts if part.strip())
    maf_first_pass_result = score_freeform_answer(
        scenario=scenario,
        answer=maf_first_pass_output,
        lane=Lane.MICROSOFT_AGENT_FRAMEWORK,
        title=f"Microsoft Agent Framework multi-agent workflow ({MAF_MODEL})",
        takeaway=(
            "MAF supplies Microsoft-native multi-agent orchestration. The same deterministic business contract "
            "still decides whether the output is grounded, complete, safe, and runbook-aligned."
        ),
        used_harness_memory=True,
    )
    display(Markdown(render_management_summary_markdown(maf_first_pass_result)))
    display(Markdown(render_executive_findings_markdown(maf_first_pass_result)))
    display(Markdown(render_rule_findings_markdown(scenario, maf_first_pass_result)))
    display(Markdown(render_memory_markdown(maf_first_pass_result.memory)))
    display(Markdown(render_model_output_html("Actual MAF multi-agent output", maf_first_pass_output)))
except Exception as exc:
    maf_first_pass_output = ""
    maf_first_pass_result = None
    display(Markdown("## MAF workflow did not complete"))
    print(type(exc).__name__, str(exc))

## 7. Add A Bounded Goal Loop Around MAF

MAF gives us agents and orchestration. Here we add the harness quality loop:

1. score the actual output deterministically
2. identify exact misses and risks
3. feed only those findings to a repair agent
4. stop when the contract passes or the attempt budget is exhausted

This is the same concept as a GoalPost or GoalLoop: the model is not allowed to be “done” just because it produced fluent text.

In [ ]:
repair_agent = Agent(
    client=client,
    name="maf-repair-agent",
    instructions=(
        "You are the repair agent in a production incident harness. "
        "Given deterministic evaluator findings, revise the plan to satisfy the quality contract. "
        "Return JSON only. Use only the incident, logs, runbook, prior memory, and accepted agent findings. "
        "Never include forbidden actions. Do not invent tools, owners, dashboards, or thresholds."
    ),
)


def findings_payload(result):
    return [finding.__dict__ for finding in evaluate_rules(scenario, result)]


async def run_maf_goal_loop(max_attempts=3):
    attempts = []
    current_output = maf_first_pass_output
    best_result = maf_first_pass_result
    best_output = current_output
    best_rank = (-1, -1)

    for attempt in range(1, max_attempts + 1):
        if best_result is None or not current_output:
            break

        current_result = score_freeform_answer(
            scenario=scenario,
            answer=current_output,
            lane=Lane.MICROSOFT_AGENT_FRAMEWORK,
            title=f"Microsoft Agent Framework goal loop candidate ({MAF_MODEL})",
            takeaway="MAF output scored by the same deterministic harness contract.",
            used_harness_memory=True,
        )
        current_rank = (1 if current_result.checks.get("safety") else 0, current_result.score)
        accepted = current_rank > best_rank
        if accepted:
            best_rank = current_rank
            best_result = current_result
            best_output = current_output

        passed = all(current_result.checks.values()) and not current_result.memory.reviewer_objections
        attempts.append({
            "attempt": attempt,
            "score": current_result.score,
            "checks": current_result.checks,
            "reviewer_objections": current_result.memory.reviewer_objections,
            "accepted": accepted,
            "passed": passed,
        })
        if passed or attempt >= max_attempts:
            break

        repair_prompt = json.dumps({
            "incident": scenario.incident,
            "logs": scenario.logs,
            "runbook": scenario.runbook,
            "prior_memory": scenario.prior_memory,
            "quality_contract": scenario.expected,
            "deterministic_findings": findings_payload(current_result),
            "candidate_output": current_output,
        }, indent=2)
        repair_response = await repair_agent.run(repair_prompt)
        current_output = as_text(repair_response)

    if best_result is not None:
        best_result.goal_loop_attempts.extend(attempts)
    return best_result, best_output, attempts

maf_result, maf_scored_output, maf_goal_attempts = await run_maf_goal_loop(max_attempts=3)

if maf_result is not None:
    display(Markdown("## MAF Goal Loop Attempts"))
    print(json.dumps(maf_goal_attempts, indent=2))
    display(Markdown(render_management_summary_markdown(maf_result)))
    display(Markdown(render_rule_findings_markdown(scenario, maf_result)))
    display(Markdown(render_model_output_html("Accepted MAF scored output", maf_scored_output)))
else:
    display(Markdown("## No scorable MAF result"))

## 8. Human-In-The-Loop Packet

For management, this is the key production behavior: the harness does not blindly execute a risky plan. If score is low, runbook alignment is missing, or safety fails, it packages the exact issue for human review.

In [ ]:
def maf_hitl_packet(result):
    findings = evaluate_rules(scenario, result)
    risks = [f for f in findings if f.severity == "risk"]
    misses = [f for f in findings if f.severity == "miss"]
    return {
        "decision": "approve" if result.score >= 85 and not risks else "human_review_required",
        "score": result.score,
        "failed_checks": [name for name, ok in result.checks.items() if not ok],
        "risks": [f.__dict__ for f in risks],
        "misses": [f.__dict__ for f in misses],
        "human_options": [
            "approve accepted plan",
            "ask repair agent to retry with listed findings",
            "escalate to incident commander",
        ],
    }

if maf_result is not None:
    display(Markdown("## MAF HITL Decision Packet"))
    print(json.dumps(maf_hitl_packet(maf_result), indent=2))

## 9. LLM Critique And Management Polish

The evaluator above is deterministic. This cell optionally asks a cheaper Ollama model to explain the deterministic findings in management language. The LLM is not scoring the output; it is only rephrasing the evidence we already computed.

In [ ]:
if maf_result is not None:
    try:
        critique = critique_groundedness_with_ollama(scenario, maf_result, model_name=SUMMARY_MODEL)
        display(Markdown("## LLM Critique Of MAF Output"))
        display(Markdown(critique))
    except Exception as exc:
        print("No LLM critique generated:", type(exc).__name__, str(exc))

    try:
        summary = summarize_result_with_ollama(scenario, maf_result, model_name=SUMMARY_MODEL)
        display(Markdown("## LLM-Polished MAF Management Summary"))
        display(Markdown(summary))
    except Exception as exc:
        print("No LLM summary generated:", type(exc).__name__, str(exc))

## 10. Optional: MAF Packaged Harness Agent

MAF also has a packaged harness agent API. This is closer to the “industry is productizing harness engineering” message: planning, todo state, file memory, tool approval, and context compaction become framework-provided pieces.

This cell is optional because the main comparison above uses explicit multi-agent orchestration for fairness with the rest of our demo.

In [ ]:
try:
    from agent_framework import create_harness_agent

    packaged_harness_agent = create_harness_agent(client)
    packaged_session = packaged_harness_agent.create_session()
    packaged_prompt = (
        "Using the supplied incident, logs, runbook, and prior memory, produce the same structured incident plan. "
        "Do not invent unsupported operational details.\n\n" + maf_task
    )
    packaged_response = await packaged_harness_agent.run(packaged_prompt, session=packaged_session)
    packaged_output = as_text(packaged_response)
    packaged_result = score_freeform_answer(
        scenario=scenario,
        answer=packaged_output,
        lane=Lane.MICROSOFT_AGENT_FRAMEWORK,
        title=f"Microsoft Agent Framework packaged harness agent ({MAF_MODEL})",
        takeaway="Packaged MAF harness agent shows the Microsoft-native path toward productized harness pieces.",
        used_harness_memory=True,
    )
    display(Markdown(render_management_summary_markdown(packaged_result)))
    display(Markdown(render_rule_findings_markdown(scenario, packaged_result)))
    display(Markdown(render_model_output_html("MAF packaged harness output", packaged_output)))
except Exception as exc:
    display(Markdown("## Packaged MAF harness cell skipped"))
    print(type(exc).__name__, str(exc))

## 11. How To Narrate This To A Microsoft-Shop Audience

- MAF is not “just another prompt wrapper.” It gives Microsoft-native primitives for multi-agent orchestration, session/history storage, middleware, HITL, and packaged harness agents.
- Our demo still matters because frameworks do not know our production truth by default: required evidence, approved runbook actions, forbidden actions, escalation thresholds, and business-quality definitions.
- The adoption message is practical: use MAF to avoid rebuilding plumbing, then encode your domain quality contract as deterministic checks, repair loops, and HITL policies.